**Importing Required Libraries**

In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from assignments.helpers import preprocess_data
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

**Loading Dataset**

In [3]:
preprocessed_data = preprocess_data()
preprocessed_data.shape

(1460, 20)

**Removal of Data**

In [4]:
# Dropping the single record with a genuinely missing BsmtExposure (basement exists, value missing)
genuine_missing = preprocessed_data["BsmtExposure"].isnull() & (preprocessed_data["TotalBsmtSF"] > 0)
print("Rows dropped:", genuine_missing.sum())
preprocessed_data = preprocessed_data[~genuine_missing].copy()
print(preprocessed_data.shape)

Rows dropped: 1
(1459, 20)


In [5]:
# Drop the two known anomalous records: very large, top-quality houses with implausibly low prices
anomalous = (preprocessed_data["GrLivArea"] > 4000) & (preprocessed_data["SalePrice"] < 300000)
print("Rows dropped:", anomalous.sum())          # expected: 2
preprocessed_data = preprocessed_data[~anomalous].copy()
print(preprocessed_data.shape)                    # expected: (1457, 20)

Rows dropped: 2
(1457, 20)


In [6]:
# Drop extreme LotArea records (huge lots with ordinary prices, no matching price signal)
extreme_lot = preprocessed_data["LotArea"] > 100000
print("Rows dropped:", extreme_lot.sum())          # expected: 4
preprocessed_data = preprocessed_data[~extreme_lot].copy()
print(preprocessed_data.shape)                      # expected: (1453, 20)

Rows dropped: 4
(1453, 20)


### **Step 4:**

##### **Split the dataset into training and test sets. Develop an appropriate preprocessing procedure for missing values, outliers and feature scaling.**

1. The dataset is split into Training(75%) and Test(25%), using a fixed random seed for reproducibility.
2.  Since, the Dependent Variable is highly right-skewed (1.88, see Step 3: Potential outlier), To avoid meaningless distribution of data in train and test sets, the split is **stratified** on `SalePrice`, binned into 5 quantile-based strata (`pd.qcut`)
3. To represent price range meaningfully, we take 5 bins to have enough strata. This way, each stratum gets (219(training) + 73(test) = 290). This generate reliable split given the skewed distribution.

In [7]:
price_bins = pd.qcut(preprocessed_data["SalePrice"], q=5, labels=False)

In [8]:
train_data, test_data = train_test_split(
    preprocessed_data,
    test_size=0.25,
    random_state=42,
    stratify=price_bins
)

In [9]:
# let's find the shape of our datasets
print(train_data.shape)
print(test_data.shape) 

(1089, 20)
(364, 20)


**Let's Identify preprocessing pipelines and their features**

In [10]:
# Column groups
median_impute_col = ["MasVnrArea"]
none_impute_ordinal_cols = ["BsmtExposure", "BsmtFinType1", "FireplaceQu", "GarageFinish"]
ordinal_only_col = ["HeatingQC"]
passthrough_cols = ["LotArea", "OverallQual", "YearBuilt", "TotalBsmtSF",
                     "GrLivArea", "FullBath", "Fireplaces", "GarageCars"]
nominal_cols = ["Neighborhood", "CentralAir", "LotConfig", "BldgType", "SaleCondition"]


In [11]:
# Category orders, these lists show the ordinal categories, add 'None' for missing features to make a new category
none_impute_ordinal_orders = [
    ["None", "No", "Mn", "Av", "Gd"],                    # BsmtExposure
    ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],  # BsmtFinType1
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],              # FireplaceQu
    ["None", "Unf", "RFn", "Fin"],                       # GarageFinish
]
heatingqc_order = [["Po", "Fa", "TA", "Gd", "Ex"]]

In [12]:
# Sub-pipelines
median_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

none_ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("encoder", OrdinalEncoder(categories=none_impute_ordinal_orders))
])

heatingqc_pipeline = Pipeline([
    ("encoder", OrdinalEncoder(categories=heatingqc_order))
])

onehot_pipeline = Pipeline([
    ("encoder", OneHotEncoder(drop="first", sparse_output=False))
])


In [14]:
# ColumnTransformer combines all of the sub-pipelines
preprocessor = ColumnTransformer([
    ("median_impute", median_pipeline, median_impute_col),
    ("none_ordinal", none_ordinal_pipeline, none_impute_ordinal_cols),
    ("heatingqc", heatingqc_pipeline, ordinal_only_col),
    ("passthrough", "passthrough", passthrough_cols),
    ("onehot", onehot_pipeline, nominal_cols),
])

**Finally, we build the complete pipeline**

In [15]:
full_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("scaling", StandardScaler())
])

In [16]:
full_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('scaling', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('median_impute', ...), ('none_ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshol

**Let's run the pipeline on our dataset**

In [17]:
X_train_raw = train_data.drop(columns=["SalePrice"])
y_train = train_data["SalePrice"]
X_test_raw = test_data.drop(columns=["SalePrice"])
y_test = test_data["SalePrice"]

X_train_transformed = full_pipeline.fit_transform(X_train_raw)
X_test_transformed = full_pipeline.transform(X_test_raw)

print("X_train_transformed shape:", X_train_transformed.shape)
print("X_test_transformed shape:", X_test_transformed.shape)

X_train_transformed shape: (1089, 52)
X_test_transformed shape: (364, 52)


## **Let's Train our models**

#### **1. SVD (Singular Value Decomposition)**

In [18]:
baseline_model = LinearRegression()
baseline_model.fit(X_train_transformed, y_train)

train_preds = baseline_model.predict(X_train_transformed)
test_preds = baseline_model.predict(X_test_transformed)

In [19]:
print("Train R²:", r2_score(y_train, train_preds))
print("Test R²:", r2_score(y_test, test_preds))
print("Train MAE:", mean_absolute_error(y_train, train_preds))
print("Test MAE:", mean_absolute_error(y_test, test_preds))
print("Train RMSE:", root_mean_squared_error(y_train, train_preds))
print("Test RMSE:", root_mean_squared_error(y_test, test_preds))

Train R²: 0.8893185936134853
Test R²: 0.8519552138826101
Train MAE: 17987.927328373222
Test MAE: 20082.50240091017
Train RMSE: 25937.810519237744
Test RMSE: 32037.122660524805


#### **2. SGD**

In [20]:
sgd_model = SGDRegressor(random_state=42)
sgd_model.fit(X_train_transformed, y_train)

sgd_train_preds = sgd_model.predict(X_train_transformed)
sgd_test_preds = sgd_model.predict(X_test_transformed)


In [21]:
print("Train R²:", r2_score(y_train, sgd_train_preds))
print("Test R²:", r2_score(y_test, sgd_test_preds))
print("Train MAE:", mean_absolute_error(y_train, sgd_train_preds))
print("Test MAE:", mean_absolute_error(y_test, sgd_test_preds))
print("Train RMSE:", root_mean_squared_error(y_train, sgd_train_preds))
print("Test RMSE:", root_mean_squared_error(y_test, sgd_test_preds))

Train R²: 0.8889205237492102
Test R²: 0.8507325439003631
Train MAE: 17876.772237698242
Test MAE: 19934.165617263185
Train RMSE: 25984.41181429236
Test RMSE: 32169.144481815976


1. **Standardize predictors:** All 52 features have been standardized via `StandardScaler`.
2. **Learning rate:** eta0 = 0.01, with learning_rate='invscaling' decays as (eta0 / iteration^0.25).
3. **Stopping criterion:** When loss improves by less than **tol = 0.001** for **n_iter_no_change = 5** consective rows.
4. **Iterations or epochs:** **n_iter_ = 25** (from the fitted model)

In [23]:
sgd_model

,"random_state random_state: int, RandomState instance, default=NoneUsed for shuffling the data, when ``shuffle`` is set to ``True``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: str, default='squared_error'The loss function to be used. The possible values are 'squared_error','huber', 'epsilon_insensitive', or 'squared_epsilon_insensitive'The 'squared_error' refers to the ordinary least squares fit.'huber' modifies 'squared_error' to focus less on getting outlierscorrect by switching from squared to linear loss past a distance ofepsilon. 'epsilon_insensitive' ignores errors less than epsilon and islinear past that; this is the loss function used in SVR.'squared_epsilon_insensitive' is the same but becomes squared loss pasta tolerance of epsilon.More details about the losses formulas can be found in the:ref:`User Guide <sgd_mathematical_formulation>`.",'squared_error'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True
,"max_iter max_iter: int, default=1000The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the ``fit`` method, and not the:meth:`partial_fit` method.Values must be in the range `[1, inf)`... versionadded:: 0.19",1000
,"tol tol: float or None, default=1e-3The stopping criterion. If it is not None, training will stopwhen (loss > best_loss - tol) for ``n_iter_no_change`` consecutiveepochs.Convergence is checked against the training loss or thevalidation loss depending on the `early_stopping` parameter.Values must be in the range `[0.0, inf)`... versionadded:: 0.19",0.001
,"shuffle shuffle: bool, default=TrueWhether or not the training data should be shuffled after each epoch.",True
,"verbose verbose: int, default=0The verbosity level.Values must be in the range `[0, inf)`.",0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-insensitive loss functions; only if `loss` is'huber', 'epsilon_insensitive', or 'squared_epsilon_insensitive'.For 'huber', determines the threshold at which it becomes lessimportant to get the prediction exactly right.For epsilon-insensitive, any differences between the current predictionand the correct label are ignored if they are less than this threshold.Values must be in the range `[0.0, inf)`.",0.1
